<a href="https://colab.research.google.com/github/Yanbing689/01-local-llm-app/blob/main/Multi_Agent_System_using_Gemini_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-generativeai

In [2]:
# see exactly which models you can use
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [3]:
import os
from google.colab import userdata
import google.generativeai as genai

def configure():
    secret_name = "GOOGLE_API_KEY"

    api_key = userdata.get(secret_name)
    if not api_key:
        raise ValueError(f"Secret '{secret_name}' not found. Please check the name in Colab's Secrets.")

    genai.configure(api_key=api_key)
    return genai.GenerativeModel('models/gemini-2.5-flash')

Agent 1: The Planner

In [4]:
def planner_agent(model, topic: str) -> list[str]:
    print("Planner Agent: Creating a research plan...")
    prompt = f"""
    You are an expert research planner. Break down this topic into exactly 3 specific questions.

    STRICT RULES:
    - Return ONLY a JSON array of strings
    - No markdown, no code blocks, no backticks
    - Each question must be complete and not contain commas
    - Exactly this format: ["question 1", "question 2", "question 3"]

    TOPIC: "{topic}"
    """
    try:
        response = model.generate_content(prompt)
        import json
        # ✅ Clean and parse as JSON for reliable splitting
        plan_text = response.text.strip()
        plan_text = plan_text.replace("```json", "").replace("```", "").strip()
        plan = json.loads(plan_text)
        print("Plan created:")
        for i, q in enumerate(plan, 1):
            print(f"   {i}. {q}")
        return plan
    except Exception as e:
        print(f"Error in Planner Agent: {e}")
        return []

Agent 2: The Search Agent

In [5]:
import time

def search_agent(model, question: str) -> str:
    print(f"Search Agent: Researching question: '{question}'...")
    for attempt in range(3):  # retry up to 3 times
        try:
            prompt = f"Provide a detailed answer to the following question: {question}"
            response = model.generate_content(prompt)
            print("   - Information found.")
            return response.text
        except Exception as e:
            if "429" in str(e):
                wait = (attempt + 1) * 60
                print(f"Rate limited. Waiting {wait} seconds... (attempt {attempt+1}/3)")
                time.sleep(wait)
            else:
                print(f"Error in Search Agent: {e}")
                return ""
    return ""

Agent 3: The Synthesizer

In [6]:
import time

def synthesizer_agent(model, topic: str, research_results: list) -> str:
    print("Synthesizer Agent: Writing the final report...")

    research_notes = ""
    for question, data in research_results:
        research_notes += f"### Question: {question}\n### Research Data:\n{data}\n\n---\n\n"

    prompt = f"""
    You are an expert research analyst. Your task is to synthesize the provided research notes
    into a comprehensive, well-structured report on the topic: "{topic}".

    The report should have an introduction, a body that covers the key findings from the notes,
    and a conclusion. Use the information from the research notes ONLY.

    ## Research Notes ##
    {research_notes}
    """
    for attempt in range(3):  # ✅ retry up to 3 times
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            if "429" in str(e):
                wait = (attempt + 1) * 15  # 15s, 30s, 45s
                print(f"Rate limited. Waiting {wait} seconds... (attempt {attempt+1}/3)")
                time.sleep(wait)
            else:
                print(f"Error in Synthesizer Agent: {e}")
                return "Error: Could not generate the final report."
    return "Error: Could not generate the final report after 3 attempts."

The Conductor: The main() Orchestratord

In [ ]:
import time

def main():
    try:
        model = configure()
    except ValueError as e:
        print(e)
        return

    print("\nHello! I am your AI Research Assistant.")
    topic = input("What topic would you like me to research today? ")

    if not topic.strip():
        print("A topic is required to begin research. Exiting.")
        return

    print(f"\nStarting research process for: '{topic}'")

    research_plan = planner_agent(model, topic)
    if not research_plan:
        print("Could not create a research plan. Exiting.")
        return

    research_results = []
    for question in research_plan:
        research_data = search_agent(model, question)
        if research_data:
            research_results.append((question, research_data))
        time.sleep(60)   # prevents hitting rate limits

    if not research_results:
        print("Could not find any information during research. Exiting.")
        return

    time.sleep(60)  # ✅ extra pause before synthesizer
    final_report = synthesizer_agent(model, topic, research_results)

    print("\n\n--- FINAL RESEARCH REPORT ---")
    print(f"## Topic: {topic}\n")
    print(final_report)
    print("--- END OF REPORT ---")

main()


Hello! I am your AI Research Assistant.
